# SepVAE — Counterfactual Latent Slider

Interactive notebook for exploring latent disentanglement in SepVAE V2.

## What this shows

Three sliders let you probe the model:

1. **α (disease strength)** — interpolates `z_disease` from 0 → mean Cardiomegaly vector.  
   If disentanglement works: only the cardiac silhouette size changes.  
   If it fails: anatomy changes too (lung fields shift, ribs move, brightness changes).

2. **Normal image index** — which Normal patient's anatomy to use (z_common source).

3. **Cardio source** — which Cardiomegaly population to compute the mean disease vector from  
   (full mean, or a single patient's z_disease for a direct swap).

## Grid layout

```
| Original Normal | Decoded (α=0, anatomy-only) | Decoded (α=slider) | Original Cardio reference |
```

The middle-left (α=0) should look identical to the original Normal.  
The middle-right (α=slider) should show the same anatomy with a progressively enlarging heart.

---

**Prerequisites:** Run from the repo root with the `jaxstack` kernel.  
**GPU:** set `CUDA_VISIBLE_DEVICES` in cell 1 before initialising JAX.

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '1'       # ← change if needed
os.environ['JAX_PLATFORMS']        = 'cuda'
os.environ['XLA_PYTHON_CLIENT_MEM_FRACTION'] = '0.35'

In [ ]:
import sys
from pathlib import Path

import jax
import jax.numpy as jnp
import numpy as np
import torch
import matplotlib
matplotlib.use('module://matplotlib_inline.backend_inline')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import ipywidgets as widgets
from IPython.display import display
from flax.serialization import msgpack_restore
from torch.utils.data import DataLoader, Subset

# Repo root on path
REPO = Path('.').resolve()
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from models.sep_vae_v2 import SepVAEV2
from datasets.VinBigData import VinBigDataPairDataset, jax_pair_collate_fn

print(f'JAX devices: {jax.devices()}')

## 1. Configuration — edit paths here

In [ ]:
CHECKPOINT = 'runs_sepvae/d3_gan_fix-20260325-143813/checkpoints/checkpoint_epoch0105.pkl'
CSV_PATH   = '/datasets/mmolefe/vinbigdata/cache_npy/train_filtered.csv'
DICOM_DIR  = '/datasets/mmolefe/vinbigdata/cache_npy'

# How many images to pre-encode for the slider pool
N_NORMAL_POOL  = 32   # Normal images to flip through
N_CARDIO_POOL  = 64   # Cardiomegaly images for mean z_disease estimation

IMG_SIZE       = 256
Z_COMMON       = 16
Z_DISEASE      = 16
ATTN_QUERY_DIM = 256
ATTN_HEADS     = 4
BBOX_QUERY_MIX = 1.0   # D3 uses pure Gaussian prior
DECODER_RES_BLOCKS = 3     # D2+ checkpoint — must match training config


## 2. Load model

In [ ]:
def _merge_params(target, source):
    if not isinstance(target, dict):
        return jnp.array(source)
    result = {}
    for k, v in target.items():
        if k in source:
            result[k] = (
                _merge_params(v, source[k])
                if isinstance(v, dict) and isinstance(source[k], dict)
                else jnp.array(source[k])
            )
        else:
            result[k] = v
    return result


model = SepVAEV2(
    z_channels_common=Z_COMMON,
    z_channels_disease=Z_DISEASE,
    query_dim=ATTN_QUERY_DIM,
    attn_heads=ATTN_HEADS,
    use_bbox_cross_attn=True,
    bbox_query_mix=BBOX_QUERY_MIX,
    decoder_res_blocks=DECODER_RES_BLOCKS,
)

dummy_x      = jnp.ones((1, IMG_SIZE, IMG_SIZE, 1))
dummy_labels = jnp.array([0])
rng          = jax.random.PRNGKey(0)
fresh_vars   = model.init(rng, dummy_x, dummy_labels, key=rng)
fresh_params = jax.tree_util.tree_map(jnp.array, fresh_vars['params'])

with open(CHECKPOINT, 'rb') as f:
    ckpt = msgpack_restore(f.read())

ckpt_params = jax.tree_util.tree_map(
    jnp.array, ckpt.get('ema_params', ckpt['vae_params'])
)
params = _merge_params(fresh_params, ckpt_params)
epoch  = int(ckpt.get('epoch', 0))
print(f'Loaded checkpoint — epoch {epoch}')


## 3. Encode image pool

In [ ]:
dataset = VinBigDataPairDataset(
    dicom_dir=DICOM_DIR,
    csv_path=CSV_PATH,
    img_size=IMG_SIZE,
    use_cache=True,
    deterministic_pairs=True,
    pair_seed=0,
)

n_total = min(max(N_NORMAL_POOL, N_CARDIO_POOL), len(dataset))
rng_np  = np.random.default_rng(42)
indices = np.sort(rng_np.choice(len(dataset), size=n_total, replace=False)).tolist()
subset  = Subset(dataset, indices)
loader  = DataLoader(
    subset,
    batch_size=16,
    shuffle=False,
    collate_fn=jax_pair_collate_fn,
    num_workers=0,
    drop_last=False,
)

all_x_normal, all_x_cardio     = [], []
all_mu_c_normal, all_mu_d_normal = [], []
all_mu_c_cardio, all_mu_d_cardio = [], []
all_bbox_cardio                  = []

print('Encoding ...', end=' ')
for batch in loader:
    x_norm   = jnp.array(batch['x_norm'].permute(0, 2, 3, 1).numpy())
    x_cardio = jnp.array(batch['x_disease1'].permute(0, 2, 3, 1).numpy())
    bbox_ca  = jnp.array(batch['bbox_disease1'].numpy())

    has_bbox = ((bbox_ca[:, 2] - bbox_ca[:, 0]) > 1e-4).astype(jnp.float32)
    bbox_z   = jnp.zeros_like(bbox_ca)
    has_z    = jnp.zeros(x_norm.shape[0], dtype=jnp.float32)

    ld_n = model.apply({'params': params}, x_norm,   method=model.encode,
                       bbox=bbox_z, has_bbox=has_z)
    ld_c = model.apply({'params': params}, x_cardio, method=model.encode,
                       bbox=bbox_ca, has_bbox=has_bbox)

    all_x_normal.append(np.array(x_norm))
    all_x_cardio.append(np.array(x_cardio))
    all_mu_c_normal.append(np.array(ld_n['common'][0]))
    all_mu_d_normal.append(np.array(ld_n['cardiomegaly'][0]))
    all_mu_c_cardio.append(np.array(ld_c['common'][0]))
    all_mu_d_cardio.append(np.array(ld_c['cardiomegaly'][0]))
    all_bbox_cardio.append(np.array(bbox_ca))

X_NORMAL   = np.concatenate(all_x_normal,    axis=0)   # (N, H, W, 1),  [-1,1]
X_CARDIO   = np.concatenate(all_x_cardio,    axis=0)
MU_C_NORM  = np.concatenate(all_mu_c_normal, axis=0)   # (N, 16, 16, 16)
MU_D_NORM  = np.concatenate(all_mu_d_normal, axis=0)
MU_C_CARD  = np.concatenate(all_mu_c_cardio, axis=0)
MU_D_CARD  = np.concatenate(all_mu_d_cardio, axis=0)
BBOX_CARD  = np.concatenate(all_bbox_cardio, axis=0)

# Limit pools
X_NORMAL   = X_NORMAL[:N_NORMAL_POOL]
MU_C_NORM  = MU_C_NORM[:N_NORMAL_POOL]
MU_D_NORM  = MU_D_NORM[:N_NORMAL_POOL]

X_CARDIO   = X_CARDIO[:N_CARDIO_POOL]
MU_C_CARD  = MU_C_CARD[:N_CARDIO_POOL]
MU_D_CARD  = MU_D_CARD[:N_CARDIO_POOL]
BBOX_CARD  = BBOX_CARD[:N_CARDIO_POOL]

# Pre-compute mean z_disease across full Cardiomegaly pool
MEAN_Z_CARD = jnp.mean(jnp.array(MU_D_CARD), axis=0)   # (16, 16, 16)

print(f'Done. Normal pool: {X_NORMAL.shape[0]}, Cardio pool: {X_CARDIO.shape[0]}')
print(f'MEAN_Z_CARD norm: {float(jnp.linalg.norm(MEAN_Z_CARD.mean((0,1)))):.4f}')

## 4. Decode helper

In [ ]:
@jax.jit
def decode_z(z_common, z_disease):
    """(H_lat, W_lat, C_c), (H_lat, W_lat, C_d) → (H, W, 1) in [0,1]"""
    z_concat = jnp.concatenate(
        [z_common[None], z_disease[None]], axis=-1
    )  # (1, 16, 16, 32)
    x_rec = model.apply({'params': params}, z_concat, method=model.decode)
    return x_rec[0]  # (H, W, 1)

## 5. Interactive slider — disease injection

**α** controls how much of the mean Cardiomegaly z_disease is injected into the Normal patient's anatomy.

- α = 0.0 → pure anatomy (Normal reconstruction)
- α = 1.0 → Normal anatomy + full mean Cardiomegaly disease latent
- α > 1.0 → exaggerated (useful to see what the latent encodes even more strongly)

The anatomy (z_common) stays fixed. Only the heart region should change.

In [ ]:
out = widgets.Output()

slider_alpha = widgets.FloatSlider(
    value=0.0, min=0.0, max=1.5, step=0.05,
    description='α (disease strength)',
    style={'description_width': '160px'},
    layout=widgets.Layout(width='600px'),
    continuous_update=False,
)
slider_normal_idx = widgets.IntSlider(
    value=0, min=0, max=len(X_NORMAL) - 1, step=1,
    description='Normal image',
    style={'description_width': '160px'},
    layout=widgets.Layout(width='500px'),
    continuous_update=False,
)
slider_cardio_idx = widgets.IntSlider(
    value=-1, min=-1, max=len(X_CARDIO) - 1, step=1,
    description='Cardio src (-1=mean)',
    style={'description_width': '160px'},
    layout=widgets.Layout(width='500px'),
    continuous_update=False,
)


def update(alpha, normal_idx, cardio_idx):
    with out:
        out.clear_output(wait=True)

        z_c = jnp.array(MU_C_NORM[normal_idx])    # anatomy from Normal

        # Disease source: mean pool or a specific Cardio patient
        if cardio_idx < 0:
            z_d_src = MEAN_Z_CARD
            src_label = f'mean of {N_CARDIO_POOL} Cardio images'
        else:
            z_d_src = jnp.array(MU_D_CARD[cardio_idx])
            src_label = f'Cardio image #{cardio_idx}'

        z_d_injected = float(alpha) * z_d_src

        # Decode
        img_anatomy  = np.array(decode_z(z_c, jnp.zeros_like(z_d_src)))
        img_injected = np.array(decode_z(z_c, z_d_injected))

        # Reference images
        orig_normal  = (X_NORMAL[normal_idx, :, :, 0] + 1.0) / 2.0
        if cardio_idx >= 0:
            orig_cardio = (X_CARDIO[cardio_idx, :, :, 0] + 1.0) / 2.0
            bbox = BBOX_CARD[cardio_idx]
        else:
            orig_cardio = None
            bbox = None

        # Plot
        n_cols = 4 if orig_cardio is not None else 3
        fig, axes = plt.subplots(1, n_cols, figsize=(n_cols * 3.2, 3.5))
        kw = dict(cmap='gray', vmin=0, vmax=1)

        axes[0].imshow(orig_normal, **kw)
        axes[0].set_title(f'Original Normal\n(image {normal_idx})', fontsize=9)

        axes[1].imshow(np.clip(img_anatomy[:, :, 0], 0, 1), **kw)
        axes[1].set_title('Anatomy-only\n(α = 0, z_disease=0)', fontsize=9)

        axes[2].imshow(np.clip(img_injected[:, :, 0], 0, 1), **kw)
        axes[2].set_title(f'Disease injected\n(α = {alpha:.2f}, src: {src_label})', fontsize=9)

        if orig_cardio is not None:
            axes[3].imshow(orig_cardio, **kw)
            axes[3].set_title(f'Reference Cardio\n(image {cardio_idx})', fontsize=9)
            # Draw bbox
            if bbox is not None and bbox[2] - bbox[0] > 1e-4:
                H, W = orig_cardio.shape
                rect = mpatches.Rectangle(
                    (bbox[0]*W, bbox[1]*H), (bbox[2]-bbox[0])*W, (bbox[3]-bbox[1])*H,
                    linewidth=1, edgecolor='lime', facecolor='none',
                )
                axes[3].add_patch(rect)

        for ax in axes:
            ax.axis('off')

        # Norm readout as title
        z_d_norm = float(jnp.linalg.norm(jnp.mean(z_d_injected, axis=(0,1))))
        fig.suptitle(
            f'z_disease norm = {z_d_norm:.3f}  |  '
            f'z_common from Normal #{normal_idx}  |  '
            f'z_disease from {src_label}  ×  α={alpha:.2f}',
            fontsize=8, y=1.01,
        )
        plt.tight_layout(pad=0.5)
        plt.show()


ui = widgets.VBox([
    slider_alpha,
    slider_normal_idx,
    slider_cardio_idx,
    out,
])

widgets.interactive_output(
    update,
    {'alpha': slider_alpha, 'normal_idx': slider_normal_idx, 'cardio_idx': slider_cardio_idx},
)

display(ui)
update(0.0, 0, -1)   # initial render

## 6. Interpolation strip — full α trajectory for one patient

Renders a horizontal strip showing the cardiac silhouette growing continuously from α=0 to α=1.5.  
Run this cell after identifying an interesting Normal/Cardio pair via the slider above.

In [ ]:
STRIP_NORMAL_IDX  = 0    # ← set from slider exploration
STRIP_CARDIO_IDX  = -1   # ← -1 for population mean, or specific index
STRIP_ALPHA_MAX   = 1.5
STRIP_N_STEPS     = 12

alphas = np.linspace(0.0, STRIP_ALPHA_MAX, STRIP_N_STEPS)

z_c = jnp.array(MU_C_NORM[STRIP_NORMAL_IDX])
z_d_src = MEAN_Z_CARD if STRIP_CARDIO_IDX < 0 else jnp.array(MU_D_CARD[STRIP_CARDIO_IDX])

frames = []
for a in alphas:
    img = np.array(decode_z(z_c, float(a) * z_d_src))
    frames.append(np.clip(img[:, :, 0], 0, 1))

fig, axes = plt.subplots(1, STRIP_N_STEPS, figsize=(STRIP_N_STEPS * 2.2, 2.8))
for i, (ax, frame, a) in enumerate(zip(axes, frames, alphas)):
    ax.imshow(frame, cmap='gray', vmin=0, vmax=1)
    ax.set_title(f'α={a:.2f}', fontsize=7)
    ax.axis('off')

src_label = f'mean Cardio' if STRIP_CARDIO_IDX < 0 else f'Cardio #{STRIP_CARDIO_IDX}'
fig.suptitle(
    f'Disease injection strip — Normal #{STRIP_NORMAL_IDX}, z_disease from {src_label}\n'
    f'Left (α=0): anatomy only.  Right (α={STRIP_ALPHA_MAX}): max disease injection.',
    fontsize=9,
)
plt.tight_layout()
plt.savefig('notebooks/interpolation_strip.png', dpi=130, bbox_inches='tight')
plt.show()
print('Saved → notebooks/interpolation_strip.png')

## 7. Swap grid — all pairs

For N Normal images and M Cardiomegaly images, decode every (z_common_normal[i], z_disease_cardio[j]) combination.  
Produces an N×M grid where rows = anatomy source, columns = disease source.

If disentanglement works:
- Each row should look like the same patient with varying heart sizes
- Each column should look like the same heart size on different anatomies

In [ ]:
SWAP_N_NORMAL = 6   # rows
SWAP_N_CARDIO = 6   # columns

fig, axes = plt.subplots(
    SWAP_N_NORMAL, SWAP_N_CARDIO + 2,   # +2 for original Normal and Cardio reference
    figsize=((SWAP_N_CARDIO + 2) * 1.8, SWAP_N_NORMAL * 1.9),
    gridspec_kw={'hspace': 0.05, 'wspace': 0.05},
)

for row, ni in enumerate(range(SWAP_N_NORMAL)):
    z_c = jnp.array(MU_C_NORM[ni])

    # Column 0: original Normal
    axes[row, 0].imshow(
        np.clip((X_NORMAL[ni, :, :, 0] + 1.0) / 2.0, 0, 1),
        cmap='gray', vmin=0, vmax=1,
    )
    if row == 0:
        axes[row, 0].set_title('Orig\nNormal', fontsize=7)
    axes[row, 0].axis('off')

    for col, ci in enumerate(range(SWAP_N_CARDIO)):
        z_d = jnp.array(MU_D_CARD[ci])
        img = np.array(decode_z(z_c, z_d))
        axes[row, col + 1].imshow(
            np.clip(img[:, :, 0], 0, 1), cmap='gray', vmin=0, vmax=1
        )
        if row == 0:
            axes[row, col + 1].set_title(f'C{ci}', fontsize=7)
        axes[row, col + 1].axis('off')

    # Last column: original Cardiomegaly (diagonal: same index as row)
    ref_idx = min(ni, SWAP_N_CARDIO - 1)
    axes[row, SWAP_N_CARDIO + 1].imshow(
        np.clip((X_CARDIO[ref_idx, :, :, 0] + 1.0) / 2.0, 0, 1),
        cmap='gray', vmin=0, vmax=1,
    )
    if row == 0:
        axes[row, SWAP_N_CARDIO + 1].set_title('Orig\nCardio', fontsize=7)
    axes[row, SWAP_N_CARDIO + 1].axis('off')

fig.suptitle(
    f'Latent swap grid — rows: Normal anatomy (z_common), columns: disease source (z_disease)\n'
    f'Each cell = Normal[row] anatomy + Cardio[col] disease',
    fontsize=9, y=1.01,
)
plt.savefig('notebooks/swap_grid.png', dpi=110, bbox_inches='tight')
plt.show()
print('Saved → notebooks/swap_grid.png')

## 8. What to look for

| Observation | Interpretation |
|-------------|----------------|
| α=0 reconstruction ≈ original Normal | Reconstruction quality intact — no degradation from the evaluation |
| Cardiac silhouette grows with α | z_disease is encoding cardiac size ✓ |
| Lung fields / ribs unchanged as α increases | z_common and z_disease are disentangled ✓ |
| Brightness or contrast shifts with α | z_disease is entangled with global intensity (leak into z_common) ✗ |
| Lung fields shift or ribs move | z_disease is entangled with anatomy ✗ |
| Swap grid rows look like same patient | z_common captures patient-specific anatomy ✓ |
| Swap grid columns look like same heart size | z_disease captures disease magnitude ✓ |

**Quantitative gate:** `z_cardio_norm_ratio` from `eval_counterfactual.py` should be ≥ 3.0 for strong disentanglement.